# 02 - Entrenamiento de los 5 modelos (corrida final, dataset completo)

## 1. Descripción

Corre los 5 modelos del plan **uno a la vez** sobre el dataset completo (`dataset_modelado_personas.csv`, `SAMPLING_MODE=False`): Time Series Split + SMOTENC por fold + cada modelo. Esta es la corrida final que alimenta la tabla comparativa de la tesis.

**Cómo usar este notebook:** ejecutar una sección de modelo a la vez.

Los modelos están ordenados de la siguiente manera: Regresión Logística -> XGBoost -> LightGBM -> CNN-LSTM -> TabNet.

**Contenido de este notebook:**

1. Descripción
2. Importaciones
3. Configuración
   - 3.1 Verificación del entorno
   - 3.2 Carga de datos
4. Selección del modelo principal
   - 4.1 Regresión Logística (EDA, Entrenamiento, Coeficientes -- sin SHAP, ver docstring de baseline_logistic_regression.py)
   - 4.2 XGBoost (EDA, Entrenamiento, Explicabilidad: SHAP global y local)
   - 4.3 LightGBM (EDA, Entrenamiento, Explicabilidad: SHAP global)
   - 4.4 CNN-LSTM (EDA, Entrenamiento, Explicabilidad: SHAP global)
   - 4.5 TabNet (EDA, Entrenamiento, Explicabilidad: SHAP global)
   - 4.6 Extensión: variables macro rezagadas (a pedido del tutor)

## 2. Importaciones

In [1]:
%run ./00_parametros_globales.ipynb

import sys
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(RAIZ_PROYECTO / "src" / "features"))
sys.path.insert(0, str(RAIZ_PROYECTO / "src" / "evaluation"))
sys.path.insert(0, str(RAIZ_PROYECTO / "src" / "models"))

from preparacion_modelado import muestrear_estratificado
from config_features import FEATURES_CATEGORICAS, FEATURES_NUMERICAS


GPU NVIDIA detectada: True
  Dispositivo: NVIDIA GeForce RTX 4090
RAIZ_PROYECTO: /home/daltamirano/democratic-satisfaction-ec
SAMPLING_MODE: False (fracción=1.0)
VENTANA_TEMPORAL_ANIOS: 3
RUTA_DATASET_PERSONAS existe: True
RUTA_PANEL_MACRO existe: True


## 3. Configuración

### 3.1 Verificación del entorno

Antes de correr cualquier modelo, confirma que las librerías necesarias están instaladas (`pip install -r requirements.txt` en tu entorno). XGBoost y LightGBM son livianas; CNN-LSTM y TabNet dependen de `torch` (y TabNet además de `pytorch-tabnet`), que son más pesadas de instalar -- mejor detectarlo aquí que a mitad de un entrenamiento.

In [2]:
paquetes = ["xgboost", "lightgbm", "torch", "pytorch_tabnet", "imblearn", "sklearn"]
for paquete in paquetes:
    try:
        __import__(paquete)
        print(f"OK      {paquete}")
    except ImportError as e:
        print(f"FALTA   {paquete}  ({e})")


OK      xgboost
OK      lightgbm
OK      torch
OK      pytorch_tabnet
OK      imblearn
OK      sklearn


### 3.2 Carga de datos

Corrida final sobre el dataset completo (`SAMPLING_MODE=False`): se cargan `dataset_modelado_personas.csv` y `panel_macro_anual.csv` sin ningún muestreo.

In [3]:
df_completo = pd.read_csv(RUTA_DATASET_PERSONAS)
panel_macro = pd.read_csv(RUTA_PANEL_MACRO)

if SAMPLING_MODE:
    df_muestra = muestrear_estratificado(df_completo, fraccion=FRACCION_MUESTRA, random_state=RANDOM_STATE)
else:
    df_muestra = df_completo

print(f"Filas dataset completo: {len(df_completo)}")
print(f"Filas usadas en esta corrida: {len(df_muestra)} (SAMPLING_MODE={SAMPLING_MODE})")
print(f"\nAños presentes: {sorted(df_muestra['anio'].unique())}")
print("\nBalance por año (%, clase 'Satisfecho'):")
print((100 * df_muestra.dropna(subset=["satisfecho_democracia"]).groupby("anio")["satisfecho_democracia"].mean()).round(1))


Filas dataset completo: 15600
Filas usadas en esta corrida: 15600 (SAMPLING_MODE=False)

Años presentes: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2023), np.int64(2024)]

Balance por año (%, clase 'Satisfecho'):
anio
2007    36.0
2008    39.5
2009    36.0
2010    50.4
2011    49.8
2013    61.2
2015    59.6
2016    42.0
2017    51.6
2018    36.5
2020    10.1
2023    12.0
2024    19.1
Name: satisfecho_democracia, dtype: float64


## 4. Selección del modelo principal

Se entrenan y comparan 5 modelos candidatos (3 baselines estadísticos/de Machine Learning -- Regresión Logística, XGBoost, LightGBM -- y 2 arquitecturas de Deep Learning -- CNN-LSTM, TabNet) bajo el mismo esquema de validación (Time Series Split + SMOTENC aplicado solo sobre `X_train` de cada fold). Cada modelo tiene su propia EDA y Entrenamiento; los 4 modelos de caja negra (XGBoost, LightGBM, CNN-LSTM, TabNet) además tienen Explicabilidad (SHAP) -- la Regresión Logística no la necesita porque sus propios coeficientes ya son directamente interpretables (ver 4.1.3).

La comparación final de los 5 (Capítulo 5 de la tesis, ver también `src/evaluation/metricas.py` y la prueba de Friedman + Wilcoxon en `src/evaluation/pruebas_estadisticas.py`) determina cuál queda como modelo principal: XGBoost obtuvo la ventaja estadísticamente más consistente en PR-AUC, aunque ninguna diferencia individual sobrevive a la corrección de Holm-Bonferroni una vez incorporados los 5 modelos (ver Sección 5.4.2 de la tesis).

### 4.1 Regresión Logística

Baseline estadístico adicional, incorporado a pedido del tutor en la segunda ronda de revisión (Sección 2.2.2 de la tesis). Se entrena primero porque es, junto con XGBoost, uno de los modelos más rápidos de los 5.

#### 4.1.1 EDA

Usa exactamente los mismos datos y las mismas features que XGBoost/LightGBM/TabNet (sección 3.2); se deja esta verificación puntual para que la sección quede autocontenida.

In [4]:
print(f"Filas usadas para entrenar: {len(df_muestra)}")
print(f"Features categóricas ({len(FEATURES_CATEGORICAS)}):", FEATURES_CATEGORICAS)
print(f"Features numéricas ({len(FEATURES_NUMERICAS)}):", FEATURES_NUMERICAS)

Filas usadas para entrenar: 15600
Features categóricas (9): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Features numéricas (39): ['resp_age', 'job_concern', 'confidence_congress_alta', 'confidence_judiciary_alta', 'confidence_church_alta', 'confidence_police_alta', 'confidence_army_alta', 'confidence_political_parties_alta', 'goods_wash_mach_bin', 'goods_car_bin', 'goods_sewage_bin', 'goods_hot_water_bin', 'tasa_participacion_global', 'tasa_desempleo', 'empleo_formal', 'empleo_informal', 'ingreso_promedio_pc', 'ingreso_promedio_laboral', 'gini_ingpc', 'pobreza_ingresos', 'pobreza_extrema_ingresos', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem', 'v2x_delibdem', 'v2x_egaldem', 'v2x_freexp_altinf', 'v2xel_frefair', 'v2xcl_rol', 'v2x_jucon', 'v2xlg_legcon', 'v2xeg_eqprotec', 'v2xeg_eqaccess', 'v2xeg_eqdr', 'v2pepwrses', 'v2pepwrsoc', 'v2pepwrgen', 'v2pepwrort', 'v2

#### 4.1.2 Entrenamiento

A diferencia de XGBoost/LightGBM, este modelo necesita codificar las categóricas one-hot y estandarizar las numéricas (ver docstring de `entrenar_evaluar_logreg`, `src/models/baseline_logistic_regression.py`). La función ya imprime la tabla de métricas por fold y el promedio al terminar.

In [5]:
import baseline_logistic_regression

# Sin ajuste de hiperparámetros (a diferencia de XGBoost/LightGBM, ver Sección 5.2 de la tesis):
# max_iter=1000, class_weight=None (SMOTENC ya corrige el balance antes de entrenar).
resultados_logreg, modelos_logreg = baseline_logistic_regression.entrenar_evaluar_logreg(df_muestra, min_anios_train=5)
baseline_logistic_regression.guardar_resultados(resultados_logreg, out_path=RUTA_TABLAS / "resultados_baseline_logreg.csv")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']


#### 4.1.3 Coeficientes (interpretabilidad, sin SHAP)

A diferencia de los 4 modelos de caja negra, la Regresión Logística no necesita SHAP: sus propios coeficientes ya indican dirección y magnitud relativa del efecto de cada variable sobre el log-odds de "Satisfecho" (ver docstring de `baseline_logistic_regression.py`). Se reportan aquí los de mayor magnitud del último fold (2024), el mismo usado en el ejemplo de SHAP local de la Sección 4.2.3 más abajo.

In [6]:
import pandas as pd

ultimo_fold_logreg = modelos_logreg[-1]
coef_logreg = pd.Series(
    ultimo_fold_logreg["modelo"].coef_[0], index=ultimo_fold_logreg["columnas"]
).sort_values(key=abs, ascending=False)
print("Top 10 coeficientes por magnitud absoluta (fold 2024):")
print(coef_logreg.head(10).round(4).to_string())

Top 10 coeficientes por magnitud absoluta (fold 2024):
econ_situation_cat_positiva                  0.8845
resp_religion_12.0                          -0.8121
resp_religion_9.0                           -0.7961
econ_situation_cat_negativa                 -0.7529
resp_religion_1.0                            0.6358
econ_situation_cat___faltante__             -0.6083
democ_supp_cat_prefiere_democracia           0.5736
democ_supp_cat_indiferente                  -0.4805
resp_economic_perception_cat___faltante__   -0.4550
resp_economic_perception_cat_positiva        0.4471


**Nota:** los coeficientes de `econ_situation_cat_positiva`/`econ_situation_cat_negativa` y `democ_supp_cat_prefiere_democracia`/`democ_supp_cat_indiferente` son estables y coherentes con el hallazgo SHAP de la Sección 4.2.3 (percepción económica y apoyo a la democracia como predictores principales). Los coeficientes grandes de `resp_religion_12.0`/`resp_religion_9.0`/`resp_religion_1.0` son menos confiables: son categorías de baja frecuencia dentro de `resp_religion`, y con pocas observaciones en el fold el one-hot puede producir coeficientes inflados sin que eso refleje un efecto real y estable (Sección 5.4.3 de la tesis discute esta limitación).

### 4.2 XGBoost

Uno de los modelos más rápidos de los 5 (junto con la Regresión Logística); se entrena primero entre los baselines de árboles por ser el más liviano.

#### 4.2.1 EDA

Verificación puntual de los datos tabulares que recibe este modelo (mismas features y mismos datos para XGBoost, LightGBM y TabNet -- ver sección 3.2).

In [7]:
print("Features categóricas:", FEATURES_CATEGORICAS)
print("Features numéricas:", FEATURES_NUMERICAS)
print(f"\nFilas usadas para entrenar: {len(df_muestra)}")
print(f"Años presentes: {sorted(df_muestra['anio'].unique())}")
print("\nBalance de 'satisfecho_democracia' (%):")
print((100 * df_muestra["satisfecho_democracia"].value_counts(normalize=True)).round(1))


Features categóricas: ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Features numéricas: ['resp_age', 'job_concern', 'confidence_congress_alta', 'confidence_judiciary_alta', 'confidence_church_alta', 'confidence_police_alta', 'confidence_army_alta', 'confidence_political_parties_alta', 'goods_wash_mach_bin', 'goods_car_bin', 'goods_sewage_bin', 'goods_hot_water_bin', 'tasa_participacion_global', 'tasa_desempleo', 'empleo_formal', 'empleo_informal', 'ingreso_promedio_pc', 'ingreso_promedio_laboral', 'gini_ingpc', 'pobreza_ingresos', 'pobreza_extrema_ingresos', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem', 'v2x_delibdem', 'v2x_egaldem', 'v2x_freexp_altinf', 'v2xel_frefair', 'v2xcl_rol', 'v2x_jucon', 'v2xlg_legcon', 'v2xeg_eqprotec', 'v2xeg_eqaccess', 'v2xeg_eqdr', 'v2pepwrses', 'v2pepwrsoc', 'v2pepwrgen', 'v2pepwrort', 'v2pepwrgeo']

Filas usadas para entrenar: 156

#### 4.2.2 Entrenamiento

La función ya imprime la tabla de métricas por fold y el promedio al terminar -- no hace falta repetirlo aquí.

In [8]:
import baseline_xgboost

# Hiperparámetros fijos (definidos en entrenar_evaluar_xgboost, src/models/baseline_xgboost.py):
# n_estimators=200, max_depth=3, learning_rate=0.05, tree_method="hist", eval_metric="aucpr".
# max_depth=3 es un ajuste conservador -- con max_depth=5 el fold 2023 sobreajustaba (ver docstring).
resultados_xgb, modelos_xgb = baseline_xgboost.entrenar_evaluar_xgboost(df_muestra, min_anios_train=5)
baseline_xgboost.guardar_resultados(resultados_xgb, out_path=RUTA_TABLAS / "resultados_baseline_xgboost.csv")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']


#### 4.2.3 Explicabilidad (SHAP)

##### SHAP (TreeSHAP) -- XGBoost

TreeSHAP (`shap.TreeExplainer`) es EXACTO para modelos de árboles -- no necesita background ni muestreo (a diferencia de KernelSHAP en CNN-LSTM/TabNet), así que se calcula sobre el fold de prueba COMPLETO de cada año. Reutiliza `modelos_xgb` (ya entrenado arriba).

In [9]:
from shap_explicabilidad import explicar_arbol, guardar_resumen_shap, graficar_resumen_shap

explicaciones_xgb = explicar_arbol(df_muestra if SAMPLING_MODE else df_completo, modelos_xgb, "xgboost")
for anio, r in explicaciones_xgb.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_xgboost_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"XGBoost -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_xgboost_{anio}.png")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
[xgboost] SHAP calculado para fold de prueba 2013 (1161 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp

##### Ejemplo de explicación LOCAL (auditoría de un encuestado puntual)

A diferencia del resumen global (impacto macro agregado), la explicación LOCAL muestra qué variables empujaron la predicción de UN encuestado en particular -- útil para auditar un perfil sociodemográfico o año específico, como pide la propuesta. Ejemplo con el primer encuestado del fold más reciente de XGBoost; cambia `idx` o `ultimo_anio_xgb` para auditar otro encuestado/año.

In [10]:
from shap_explicabilidad import explicar_perfil_local

ultimo_anio_xgb = max(explicaciones_xgb.keys())
perfil_local = explicar_perfil_local(
    explicaciones_xgb[ultimo_anio_xgb]["shap_values"],
    explicaciones_xgb[ultimo_anio_xgb]["X_test"],
    FEATURES_CATEGORICAS + FEATURES_NUMERICAS,
    idx=0,
)
print(f"Explicación LOCAL del primer encuestado del fold {ultimo_anio_xgb} (XGBoost):")
print(perfil_local.to_string(index=False))


Explicación LOCAL del primer encuestado del fold 2024 (XGBoost):
                          feature     valor_observado  shap_value
                    empleo_formal           41.679028   -0.632386
                   democ_supp_cat prefiere_democracia    0.279588
         confidence_congress_alta                 0.0   -0.147225
        confidence_judiciary_alta                 0.0   -0.122877
             confidence_army_alta                 1.0    0.109067
                   v2xeg_eqprotec               0.255   -0.095271
                      job_concern                 1.0   -0.092208
confidence_political_parties_alta                 0.0   -0.087124
                    v2x_polyarchy               0.651   -0.085163
                     v2x_delibdem                0.35   -0.059586


### 4.3 LightGBM

Corre esta sección solo después de revisar los resultados de XGBoost arriba.

#### 4.3.1 EDA

LightGBM usa exactamente los mismos datos y las mismas features que XGBoost (sección 4.1.1); se deja esta verificación puntual para que la sección quede autocontenida.

In [11]:
print(f"Filas usadas para entrenar: {len(df_muestra)}")
print(f"Features categóricas ({len(FEATURES_CATEGORICAS)}):", FEATURES_CATEGORICAS)
print(f"Features numéricas ({len(FEATURES_NUMERICAS)}):", FEATURES_NUMERICAS)


Filas usadas para entrenar: 15600
Features categóricas (9): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Features numéricas (39): ['resp_age', 'job_concern', 'confidence_congress_alta', 'confidence_judiciary_alta', 'confidence_church_alta', 'confidence_police_alta', 'confidence_army_alta', 'confidence_political_parties_alta', 'goods_wash_mach_bin', 'goods_car_bin', 'goods_sewage_bin', 'goods_hot_water_bin', 'tasa_participacion_global', 'tasa_desempleo', 'empleo_formal', 'empleo_informal', 'ingreso_promedio_pc', 'ingreso_promedio_laboral', 'gini_ingpc', 'pobreza_ingresos', 'pobreza_extrema_ingresos', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem', 'v2x_delibdem', 'v2x_egaldem', 'v2x_freexp_altinf', 'v2xel_frefair', 'v2xcl_rol', 'v2x_jucon', 'v2xlg_legcon', 'v2xeg_eqprotec', 'v2xeg_eqaccess', 'v2xeg_eqdr', 'v2pepwrses', 'v2pepwrsoc', 'v2pepwrgen', 'v2pepwrort', 'v2

#### 4.3.2 Entrenamiento

La función ya imprime la tabla de métricas por fold y el promedio al terminar.

In [12]:
import baseline_lightgbm

# Hiperparámetros fijos (definidos en entrenar_evaluar_lightgbm, src/models/baseline_lightgbm.py):
# n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05, objective="binary".
# Punto de partida razonable, no el resultado de una búsqueda exhaustiva (ver docstring).
resultados_lgbm, modelos_lgbm = baseline_lightgbm.entrenar_evaluar_lightgbm(df_muestra, min_anios_train=5)
baseline_lightgbm.guardar_resultados(resultados_lgbm, out_path=RUTA_TABLAS / "resultados_baseline_lightgbm.csv")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']


#### 4.3.3 Explicabilidad (SHAP)

##### SHAP (TreeSHAP) -- LightGBM

Igual que XGBoost: TreeSHAP es exacto (no necesita background ni muestreo), así que se calcula sobre el fold de prueba COMPLETO de cada año, sin ningún recorte.

In [13]:
from shap_explicabilidad import explicar_arbol, guardar_resumen_shap, graficar_resumen_shap

explicaciones_lgbm = explicar_arbol(df_muestra if SAMPLING_MODE else df_completo, modelos_lgbm, "lightgbm")
for anio, r in explicaciones_lgbm.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_lightgbm_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"LightGBM -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_lightgbm_{anio}.png")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
[lightgbm] SHAP calculado para fold de prueba 2013 (1161 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_sup

/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Filas antes de SMOTENC: 6856  ->  después: 7454
[lightgbm] SHAP calculado para fold de prueba 2015 (1174 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 47.68%, clase 0 (No Satisfecho): 52.32%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 47.7%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Filas antes de SMOTENC: 8030  ->  después: 8402


...

Balance de este fold -- clase 1 (Satisfecho): 43.07%, clase 0 (No Satisfecho): 56.93%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 12717  ->  después: 14480


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[lightgbm] SHAP calculado para fold de prueba 2023 (1186 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 40.42%, clase 0 (No Satisfecho): 59.58%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 13903  ->  después: 16568


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[lightgbm] SHAP calculado para fold de prueba 2024 (1196 encuestados reales, no sintéticos).
--- fold 2013 ---
                              feature  importancia_media_abs_shap
0                  econ_situation_cat                    0.514682
1                      democ_supp_cat                    0.398577
2            confidence_congress_alta                    0.325094
3        resp_economic_perception_cat                    0.299094
4   confidence_political_parties_alta                    0.200066
5           tasa_participacion_global                    0.148138
6              confidence_police_alta                    0.147293
7                      tasa_desempleo                    0.138258
8                            resp_age                    0.132482
9                         job_concern                    0.120090
10               confidence_army_alta                    0.118783
11          confidence_judiciary_alta                    0.112573
12                      ideolog

### 4.4 CNN-LSTM

Requiere `torch` instalado (ver verificación del entorno en la sección 3.1). Es notablemente más pesado que los 2 anteriores; esta corrida final entrena con 15 épocas (valor optimizado del script, ver docstring de `entrenar_evaluar_cnn_lstm`).

#### 4.4.1 EDA

A diferencia de XGBoost/LightGBM/TabNet (solo tabulares), CNN-LSTM además usa `panel_macro_anual.csv` para construir la ventana temporal (rama CNN+LSTM).

In [14]:
print(f"panel_macro shape: {panel_macro.shape}  (una fila por año, 2007-2024, sin huecos)")
print(f"Ventana temporal configurada: {VENTANA_TEMPORAL_ANIOS} años hacia atrás por persona")
print(f"Filas en los datos tabulares (rama estática): {len(df_muestra)}")
print(f"Años presentes: {sorted(df_muestra['anio'].unique())}")


panel_macro shape: (18, 41)  (una fila por año, 2007-2024, sin huecos)
Ventana temporal configurada: 3 años hacia atrás por persona
Filas en los datos tabulares (rama estática): 15600
Años presentes: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2023), np.int64(2024)]


#### 4.4.2 Entrenamiento

La función ya imprime la tabla de métricas por fold y el promedio al terminar.

In [15]:
import cnn_lstm

# Hiperparámetros fijos (definidos en entrenar_evaluar_cnn_lstm, src/models/cnn_lstm.py):
# batch_size=64, lr=1e-3, weight_decay=1e-4, canales_cnn=16, hidden_lstm=32, dim_embedding=4.
# n_epochs=15 y weight_decay=1e-4 son un ajuste conservador documentado, no una búsqueda de
# hiperparámetros (ver docstring: con n_epochs=30 el modelo sobreajustaba).
resultados_cnn, modelos_cnn = cnn_lstm.entrenar_evaluar_cnn_lstm(
    df_muestra, panel_macro,
    n_epochs=5 if SAMPLING_MODE else 15,  # 15 = valor por defecto ya optimizado del script (ver docstring)
)
cnn_lstm.guardar_resultados(resultados_cnn, out_path=RUTA_TABLAS / "resultados_cnn_lstm.csv")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']


#### 4.4.3 Explicabilidad (SHAP)

##### SHAP (KernelSHAP) -- CNN-LSTM

A diferencia de TreeSHAP, KernelSHAP no es exacto y necesita un background/muestreo -- ver la nota de costo computacional en la sección 4.5 del `GUIA_EJECUCION.md`.

In [16]:
from shap_explicabilidad import explicar_cnn_lstm, guardar_resumen_shap, graficar_resumen_shap

explicaciones_cnn = explicar_cnn_lstm(df_muestra if SAMPLING_MODE else df_completo, panel_macro, modelos_cnn)
for anio, r in explicaciones_cnn.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_cnn_lstm_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"CNN-LSTM -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_cnn_lstm_{anio}.png")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
[cnn_lstm] SHAP (KernelSHAP) calculado para fold de prueba 2013 (300 encuestados reales).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_c

### 4.5 TabNet

Requiere `torch` + `pytorch-tabnet`. Igual que el CNN-LSTM, esta corrida final entrena con 100 épocas (valor por defecto del script).

#### 4.5.1 EDA

TabNet usa los mismos datos y las mismas features que XGBoost/LightGBM (tabular, sin ventana temporal).

In [17]:
print(f"Filas usadas para entrenar: {len(df_muestra)}")
print(f"Features categóricas ({len(FEATURES_CATEGORICAS)}):", FEATURES_CATEGORICAS)
print(f"Features numéricas ({len(FEATURES_NUMERICAS)}):", FEATURES_NUMERICAS)


Filas usadas para entrenar: 15600
Features categóricas (9): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Features numéricas (39): ['resp_age', 'job_concern', 'confidence_congress_alta', 'confidence_judiciary_alta', 'confidence_church_alta', 'confidence_police_alta', 'confidence_army_alta', 'confidence_political_parties_alta', 'goods_wash_mach_bin', 'goods_car_bin', 'goods_sewage_bin', 'goods_hot_water_bin', 'tasa_participacion_global', 'tasa_desempleo', 'empleo_formal', 'empleo_informal', 'ingreso_promedio_pc', 'ingreso_promedio_laboral', 'gini_ingpc', 'pobreza_ingresos', 'pobreza_extrema_ingresos', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem', 'v2x_delibdem', 'v2x_egaldem', 'v2x_freexp_altinf', 'v2xel_frefair', 'v2xcl_rol', 'v2x_jucon', 'v2xlg_legcon', 'v2xeg_eqprotec', 'v2xeg_eqaccess', 'v2xeg_eqdr', 'v2pepwrses', 'v2pepwrsoc', 'v2pepwrgen', 'v2pepwrort', 'v2

#### 4.5.2 Entrenamiento

La función ya imprime la tabla de métricas por fold y el promedio al terminar.

In [18]:
import tabnet_model

# Hiperparámetros fijos (definidos en entrenar_evaluar_tabnet, src/models/tabnet_model.py):
# batch_size=256, virtual_batch_size=64, cat_emb_dim=4, n_d=8, n_a=8, n_steps=3, gamma=1.3.
# No se usa eval_set para early stopping (evita filtrar información del test, ver docstring).
resultados_tabnet, modelos_tabnet = tabnet_model.entrenar_evaluar_tabnet(
    df_muestra,
    max_epochs=10 if SAMPLING_MODE else 100,  # 100 = valor por defecto del script
)
tabnet_model.guardar_resultados(resultados_tabnet, out_path=RUTA_TABLAS / "resultados_tabnet.csv")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 6856  ->  después: 7454


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Balance de este fold -- clase 1 (Satisfecho): 47.68%, clase 0 (No Satisfecho): 52.32%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 47.7%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 8030  ->  después: 8402


...

Balance de este fold -- clase 1 (Satisfecho): 43.07%, clase 0 (No Satisfecho): 56.93%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 12717  ->  después: 14480


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Balance de este fold -- clase 1 (Satisfecho): 40.42%, clase 0 (No Satisfecho): 59.58%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 13903  ->  después: 16568


/home/daltamirano/democratic-satisfaction-ec/.venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)



[tabnet_model] Resultados por fold:
 anio_test  n_anios_train  n_train  n_test  accuracy  f1_macro  f1_weighted  f1_satisfecho  pr_auc  tiempo_entrenamiento_seg
      2013              5     5695    1161    0.6529    0.5486       0.5974         0.7656  0.6748                     18.66
      2015              6     6856    1174    0.6261    0.6191       0.6290         0.6707  0.7279                     22.71
      2016              7     8030    1174    0.6244    0.6226       0.6267         0.5965  0.5842                     24.22
      2017              8     9204    1183    0.6281    0.6239       0.6226         0.5841  0.6924                     29.68
      2018              9    10387    1166    0.6398    0.6157       0.6416         0.5195  0.5274                     32.81
      2020             10    11553    1164    0.8651    0.5764       0.8552         0.2266  0.2312                     37.15
      2023             11    12717    1186    0.8769    0.5271       0.8364         0.12

#### 4.5.3 Explicabilidad (SHAP)

##### SHAP (KernelSHAP) -- TabNet

In [19]:
from shap_explicabilidad import explicar_tabnet

explicaciones_tabnet = explicar_tabnet(df_muestra if SAMPLING_MODE else df_completo, modelos_tabnet)
for anio, r in explicaciones_tabnet.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_tabnet_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"TabNet -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_tabnet_{anio}.png")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
[tabnet] SHAP (KernelSHAP) calculado para fold de prueba 2013 (300 encuestados reales).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat

### 4.6 Extensión: variables macro rezagadas

Componente **temporal** también en los modelos no secuenciales (XGBoost, LightGBM, TabNet), agregando rezagos del estilo `empleo_t | empleo_t-1 | empleo_t-3`.

`src/features/rezagos_macro.py` agrega columnas `{variable}_lag1` y `{variable}_lag3` para 6 indicadores clave (`tasa_desempleo`, `gini_ingpc`, `ingreso_promedio_pc`, `pobreza_ingresos`, `v2x_polyarchy`, `v2x_libdem`), tomadas de `panel_macro_anual.csv` (ENEMDU/V-Dem, serie anual SIN huecos -- por eso los huecos de Latinobarómetro no afectan a estos rezagos).

Abajo se comparan los resultados de XGBoost **con** y **sin** estos rezagos, sobre los mismos datos ya cargados en la sección 3.2.

In [20]:
from rezagos_macro import agregar_rezagos_a_dataset

# panel_macro ya se cargó en la sección 3.2 -- se reutiliza, no hace falta releerlo.
df_muestra_ext, columnas_lag = agregar_rezagos_a_dataset(df_muestra, panel_macro)
print(f"Columnas de rezago agregadas: {columnas_lag}")
print(f"Shape datos: {df_muestra.shape}  ->  con rezagos: {df_muestra_ext.shape}")


Columnas de rezago agregadas: ['tasa_desempleo_lag1', 'tasa_desempleo_lag3', 'gini_ingpc_lag1', 'gini_ingpc_lag3', 'ingreso_promedio_pc_lag1', 'ingreso_promedio_pc_lag3', 'pobreza_ingresos_lag1', 'pobreza_ingresos_lag3', 'v2x_polyarchy_lag1', 'v2x_polyarchy_lag3', 'v2x_libdem_lag1', 'v2x_libdem_lag3']
Shape datos: (15600, 71)  ->  con rezagos: (15600, 83)


In [21]:
import baseline_xgboost
import importlib
importlib.reload(baseline_xgboost)

# Sin rezagos (features_numericas por defecto, ya corrido en la sección 4.1 -- se repite aquí para comparar lado a lado)
resultados_sin_lags, _ = baseline_xgboost.entrenar_evaluar_xgboost(df_muestra, min_anios_train=5, random_state=RANDOM_STATE)

# Con rezagos: FEATURES_NUMERICAS + columnas_lag
resultados_con_lags, _ = baseline_xgboost.entrenar_evaluar_xgboost(
    df_muestra_ext, features_numericas=FEATURES_NUMERICAS + columnas_lag, min_anios_train=5, random_state=RANDOM_STATE
)

print(f"PR-AUC promedio SIN rezagos: {resultados_sin_lags['pr_auc'].mean():.4f}")
print(f"PR-AUC promedio CON rezagos: {resultados_con_lags['pr_auc'].mean():.4f}")


[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTENC: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTENC va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']


**Nota:** esta comparación corre sobre el dataset completo (`SAMPLING_MODE=False`) -- el PR-AUC impreso arriba (con y sin rezagos) es el valor que debe citarse en la tesis para la decisión sobre incluir o no las variables macro rezagadas en los modelos no secuenciales.